Final aggregation script
------------------------
This script uses the merged LinkedIn LLM dataset with classification and readability output on post-level, to compile the variables in various way and add them to the LSEG dataset on company-level. It includes calculating count of ESG posts compared to total posts, and calculating the GAP score using a new Weighted ESG Score.

In [1]:
import os
import pandas as pd
import numpy as np

configuration:

In [ ]:
ROOT = "C:--FILL IN YOUR ROOT FOLDER--"  # root project folder
LINKEDIN_DIR = "."
LINKEDIN_FILE = "LinkedIn LLM Analysis output.csv"
LSEG_DIR = f"{ROOT}/Data Collection/LinkedIn"
LSEG_FILE = "LSEG 2025 Dataset - LinkedIns.csv" # LSEG dataset filtered on LinkedIn profile availability

OUTPUT_DIR = "."
OUTPUT_FILE = "Final Dataset.csv"

# Classification columns to represent post themes
CLSF_COLS = [
    "Cat_E", "Cat_S", "Cat_G", "Cat_N",
    "Resource_Use_active", "Emissions_active", "Innovation_active",
    "Workforce_active", "Human_Rights_active", "Community_active",
    "Product_Responsibility_active", "Management_active",
    "Shareholders_active", "CSR_Strategy_active"
]

# Readability columns to aggregate under different filters
RDBL_COLS = [
    "Gunning_Fog", "A_01_score", "A_02_score",
    "B_01_score", "B_02_score", "B_03_score", 
    "C_01_score", "C_02_score", "C_04_score", "C_05_score",
    "C_06_score", "D_01_score", "L_01_score", "Crit_A",
    "Crit_B", "Crit_C", "Crit_D", "Crit_L",
    "RDBL"  # The average readability score (averaged across all subcriteria)
]

Main script:

In [ ]:
print("Loading datasets...")
# Construct full file paths
linkedin_path = os.path.join(LINKEDIN_DIR, LINKEDIN_FILE)
lseg_path = os.path.join(LSEG_DIR, LSEG_FILE)

# Load datasets (Use pd.read_excel if your files are Excel format)
df1 = pd.read_csv(linkedin_path)
df2 = pd.read_csv(lseg_path)

print("Preprocessing Dataset 1...")
# Rename base word and sentence counts, these won't be used in final regression
df1 = df1.rename(columns={
    "word_count": "WORDS_per_post",
    "sentence_count": "SENTENCES_per_post"
})

# Convert "Yes"/"No" to 1/0 (integers) to easily compute percentages using mean()
for col in CLSF_COLS:
    df1[col] = df1[col].map({"Yes": 1, "No": 0}).fillna(0).astype(int)

# Create the ESG helper column (1 if E, S, or G is 1, else 0)
df1["ESG"] = ((df1["Cat_E"] == 1) | (df1["Cat_S"] == 1) | (df1["Cat_G"] == 1)).astype(int)

# Create Cat_ESG (relative post count helper, which is equal to ESG)
df1["Cat_ESG"] = df1["ESG"]

# Handle N/A values in readability columns so they are skipped in mean calculations
for col in RDBL_COLS:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')

# --- AGGREGATION ---
print("Starting aggregation processes...")

# 1. Volume metrics per company: POSTS (total # posts), POSTSln, WORDS (total # words), WORDSln_total, WORDS_per_post (average # words per post), SENTENCES_per_post (average # sentences per post)
volume_df = df1.groupby("Company Name").agg(
    POSTS=("WORDS_per_post", "size"),
    WORDS=("WORDS_per_post", "sum"),
    WORDS_per_post=("WORDS_per_post", "mean"),
    SENTENCES_per_post=("SENTENCES_per_post", "mean")
).reset_index()

# Calculate natural logarithms
volume_df["POSTSln"] = np.log(volume_df["POSTS"])
volume_df["WORDSln"] = np.log(volume_df["WORDS"])

# 2. Relative Post Counts (_posts)
# Create a temporary dataframe for post percentages
all_themes = ["Cat_ESG"] + CLSF_COLS
post_rules = {col: "mean" for col in all_themes}
posts_percentage_df = df1.groupby("Company Name").agg(post_rules).reset_index()

# Rename columns with '_posts' suffix and handle variations
post_rename_dict = {}
for col in all_themes:
    if col == "Cat_N":
        # Cat_ESG_posts is defined as 1 - Cat_N_posts, but we can also map Cat_N directly
        post_rename_dict[col] = "Cat_N_posts"
    elif "_active" in col:
        post_rename_dict[col] = col.replace("_active", "_posts")
    else:
        post_rename_dict[col] = f"{col}_posts"

posts_percentage_df = posts_percentage_df.rename(columns=post_rename_dict)

# Override Cat_ESG_posts explicitly to guarantee mathematical consistency (1 - Cat_N_posts)
# (Though mean of ESG dynamic helper yields the same result)
posts_percentage_df["Cat_ESG_posts"] = 1 - posts_percentage_df["Cat_N_posts"]

# 3. Relative Word Counts (_words)
# To find the relative word counts, we multiply the dummy (1/0) by the WORDS_per_post of that row
word_percentage_data = {"Company Name": volume_df["Company Name"]}
for col in all_themes:
    # Word count dedicated to this specific category per post
    df1[f"{col}_word_vol"] = df1[col] * df1["WORDS_per_post"]

# Sum up the category word volumes per company
word_vol_cols = [f"{col}_word_vol" for col in all_themes]
words_sum_df = df1.groupby("Company Name")[word_vol_cols].sum().reset_index()

# Merge with volume_df to get access to 'WORDS' total for division
words_percentage_df = pd.merge(words_sum_df, volume_df[["Company Name", "WORDS"]], on="Company Name")

# Calculate the relative word metrics
for col in all_themes:
    word_suffix = "Cat_N_words" if col == "Cat_N" else (col.replace("_active", "_words") if "_active" in col else f"{col}_words")
    words_percentage_df[word_suffix] = words_percentage_df[f"{col}_word_vol"] / words_percentage_df["WORDS"]

# Drop helper columns from the words dataframe
words_percentage_df = words_percentage_df.drop(columns=word_vol_cols + ["WORDS"])
# Ensure Cat_ESG_words matches the remaining non-N variance
words_percentage_df["Cat_ESG_words"] = 1 - words_percentage_df["Cat_N_words"]

# Combine all textual aggregations into one master frame
aggregated_df = pd.merge(volume_df, posts_percentage_df, on="Company Name", how="inner")
aggregated_df = pd.merge(aggregated_df, words_percentage_df, on="Company Name", how="inner")

# --- READABILITY AGGREGATIONS (FILTERED) ---
def aggregate_filtered_readability(df, filter_condition, suffix):
    filtered_df = df[filter_condition]
    read_agg = filtered_df.groupby("Company Name")[RDBL_COLS].mean().reset_index()
    rename_dict = {col: f"{col}{suffix}" for col in RDBL_COLS}
    return read_agg.rename(columns=rename_dict)

# Standard readability scores
standard_read = df1.groupby("Company Name")[RDBL_COLS].mean().reset_index()
aggregated_df = pd.merge(aggregated_df, standard_read, on="Company Name", how="left")

# ESG filtered readability scores
esg_read = aggregate_filtered_readability(df1, df1["ESG"] == 1, "_ESG")
aggregated_df = pd.merge(aggregated_df, esg_read, on="Company Name", how="left")

# Main Categories (E, S, G, N) filtered readability scores
for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
    suffix = f"_{cat.split('_')[1]}"
    cat_read = aggregate_filtered_readability(df1, df1[cat] == 1, suffix)
    aggregated_df = pd.merge(aggregated_df, cat_read, on="Company Name", how="left")

# --- MERGING WITH DATASET 2 ---
print("Merging aggregated data with Dataset 2...")
final_df = pd.merge(df2, aggregated_df, on="Company Name", how="left")

# CRITICAL STEP: Drop rows for companies that don't have any LinkedIn posts
# This sets up our filtered sample space before doing Z-score standardization
final_df = final_df.dropna(subset=["WORDS_per_post"])

# --- ADVANCED METRICS ---
print("Calculating advanced metrics: Z-scores, Weighted Scores, and GAP...")

# Helper function to generate z-scores safely
def z_score(series):
    return (series - series.mean()) / series.std()

# 1. Weighted ESG Scores (_posts and _words)
# Pillar Weights
final_df["ESG Weighted Score_posts"] = (
    final_df["Environmental Pillar Score"] * final_df["Cat_E_posts"] +
    final_df["Social Pillar Score"] * final_df["Cat_S_posts"] +
    final_df["Governance Pillar Score"] * final_df["Cat_G_posts"]
)

final_df["ESG Weighted Score_words"] = (
    final_df["Environmental Pillar Score"] * final_df["Cat_E_words"] +
    final_df["Social Pillar Score"] * final_df["Cat_S_words"] +
    final_df["Governance Pillar Score"] * final_df["Cat_G_words"]
)

# 2. Z-scores for (Weighted) LSEG Scores
final_df[f"ESG Score_z"] = z_score(final_df["ESG Score"])
final_df[f"ESG Weighted Score_posts_z"] = z_score(final_df["ESG Weighted Score_posts"])
final_df[f"ESG Weighted Score_words_z"] = z_score(final_df["ESG Weighted Score_words"])

# 4. GAP Variable Calculation
# using Weighted ESG Score (_words) and ESG Score Z-Scores
final_df["GAP"] = final_df["ESG Weighted Score_words_z"] - final_df["ESG Score_z"]

# CONTR score transformation
# transforms the ESG Controversy Score to a binary variable for logistic regression
final_df["CONTR"] = np.where(final_df["ESG Controversies Score"] == 100, 0, 1)

# --- MOVE COLUMNS ---
# To make sure the control, independent and dependent variables are infront
target_cols = ["WORDSln", "CONTR", "GAP", "RDBL"]
remaining_cols = [col for col in final_df.columns if col not in target_cols]
new_column_order = remaining_cols[:12] + target_cols + remaining_cols[12:]
final_df = final_df[new_column_order]

# --- EXPORT ---
# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

# Save the final matrix
final_df.to_csv(output_path, index=False)
print(f"Success! The comprehensive dataset has been saved to: {output_path}")